In [1]:
import pandas as pd
from datetime import time
from datetime import datetime

In [ ]:
CSV_SAVE = False

In [3]:
features_df = pd.read_csv("../../data/assignement_01/processed/selected_vars_VER.csv")

In [4]:
features_df["Driver"] = features_df["Driver"].astype("string")
features_df["Time"] = features_df["Time"].astype("string")

In [5]:
features_df["Time"] = features_df["Time"].apply(lambda a: a.split(" ")[2])
features_df["Time"] = features_df["Time"].apply(lambda a: a[:-3])
features_df["Time"] = features_df["Time"].apply(lambda a: a.strip())
features_df.loc[0, "Time"] = "00:00:00.000"

In [6]:
#features_df["Time"] = pd.to_datetime(features_df["Time"], format="%H:%M:%S.%f").dt.time

In [7]:
features_df["Time"] = features_df["Time"].apply(lambda a: time.strptime(a, "%H:%M:%S.%f")).astype("string")

In [8]:
def calc_ms_time(time_str):
    nrs = time_str.split(":")
    ms_in_s, ms_in_m, ms_in_h = float(nrs[2])*1_000, int(nrs[1])*60_000, int(nrs[0])*3_600_000
    return ms_in_s + ms_in_m + ms_in_h

In [9]:
features_df["Time_ms"] = features_df.loc[1:, :]["Time"].apply(lambda a: calc_ms_time(a))

In [10]:
df_recalculated_time = features_df.loc[features_df["LapNumber"]==1,:]
df_recalculated_time

,Driver,RPM,nGear,Speed,LapNumber,Time,Time_ms
0,VER,9973,1,0,1.0,00:00:00,NaN
1,VER,9963,1,0,1.0,00:00:00.048000,48.0
2,VER,9859,1,0,1.0,00:00:00.118000,118.0
3,VER,9755,1,0,1.0,00:00:00.288000,288.0
4,VER,9125,1,1,1.0,00:00:00.358000,358.0
...,...,...,...,...,...,...,...
738,VER,11002,7,274,1.0,00:01:37.077000,97077.0
739,VER,11055,7,275,1.0,00:01:37.207000,97207.0
740,VER,11035,7,276,1.0,00:01:37.237000,97237.0
741,VER,11015,7,277,1.0,00:01:37.367000,97367.0


In [11]:
lap_grps = features_df.groupby("LapNumber")

for lap_grp in lap_grps:
    sc, lap_grp = lap_grp
    if sc == 1.0:
        continue
    lap_grp = lap_grp.reset_index(drop=True)
    lap_grp.loc[0, "*NewTime"] = 0
    lap_grp["*Time_ms_shifted"] = lap_grp["Time_ms"].shift(periods=1)
    lap_grp.loc[1:, "*NewTime"] = lap_grp.loc[1:, "Time_ms"] - lap_grp.loc[1:, "*Time_ms_shifted"]
    lap_grp["Time_ms"] = lap_grp["*NewTime"]
    lap_grp["Time_ms"] = lap_grp["Time_ms"].cumsum()
    lap_grp.drop(["*Time_ms_shifted", "*NewTime"], inplace=True, axis=1)
    df_recalculated_time = pd.concat([df_recalculated_time, lap_grp])

In [12]:
df_recalculated_time = df_recalculated_time.reset_index(drop=True)
df_recalculated_time.loc[0, "Time_ms"] = 0
df_recalculated_time

,Driver,RPM,nGear,Speed,LapNumber,Time,Time_ms
0,VER,9973,1,0,1.0,00:00:00,0.0
1,VER,9963,1,0,1.0,00:00:00.048000,48.0
2,VER,9859,1,0,1.0,00:00:00.118000,118.0
3,VER,9755,1,0,1.0,00:00:00.288000,288.0
4,VER,9125,1,1,1.0,00:00:00.358000,358.0
...,...,...,...,...,...,...,...
41670,VER,10997,7,276,57.0,01:31:44.464000,94240.0
41671,VER,11059,7,277,57.0,01:31:44.596000,94372.0
41672,VER,11121,7,279,57.0,01:31:44.744000,94520.0
41673,VER,11044,7,277,57.0,01:31:44.916000,94692.0


In [13]:
#df_recalculated_time.to_csv(
#    "../../data/assignement_01/processed/preprocessed_VER.csv",
#    index=False,
#)

In [14]:
df_recalculated_time.iloc[740:750, :]

,Driver,RPM,nGear,Speed,LapNumber,Time,Time_ms
740,VER,11035,7,276,1.0,00:01:37.237000,97237.0
741,VER,11015,7,277,1.0,00:01:37.367000,97367.0
742,VER,11057,7,277,1.0,00:01:37.437000,97437.0
743,VER,11100,7,278,2.0,00:01:37.607000,0.0
744,VER,11109,7,279,2.0,00:01:37.697000,90.0
745,VER,11119,7,280,2.0,00:01:37.807000,200.0
746,VER,11141,7,280,2.0,00:01:37.897000,290.0
747,VER,11163,7,281,2.0,00:01:38.007000,400.0
748,VER,11163,7,281,2.0,00:01:38.177000,570.0
749,VER,11163,7,282,2.0,00:01:38.207000,600.0


In [15]:
tt = df_recalculated_time.groupby("LapNumber")

z = 0

for t in tt:
    sc, df = t
    z += df["Time_ms"].values[-1]

In [16]:
z

np.float64(5495929.0)

In [17]:
df_recalculated_time["Time_ms"] = df_recalculated_time["Time_ms"].astype("string")

In [18]:
df_recalculated_time[["LapNumber", "Time_ms"]]

,LapNumber,Time_ms
0,1.0,0.0
1,1.0,48.0
2,1.0,118.0
3,1.0,288.0
4,1.0,358.0
...,...,...
41670,57.0,94240.0
41671,57.0,94372.0
41672,57.0,94520.0
41673,57.0,94692.0


In [19]:
def adjust_pitstop_lbl(df, *, lapNumber: list[float]):
    for lp_nr in lapNumber:
        df.loc[df["LapNumber"] == str(lp_nr), "LapNumber"]=df.loc[df["LapNumber"] == str(lp_nr), "LapNumber"].apply(lambda a: str(a) + " PitStop")

#adjust_pitstop_lbl(df_recalculated_time, lapNumber=[18.0,38.0])

In [20]:
df_recalculated_time

,Driver,RPM,nGear,Speed,LapNumber,Time,Time_ms
0,VER,9973,1,0,1.0,00:00:00,0.0
1,VER,9963,1,0,1.0,00:00:00.048000,48.0
2,VER,9859,1,0,1.0,00:00:00.118000,118.0
3,VER,9755,1,0,1.0,00:00:00.288000,288.0
4,VER,9125,1,1,1.0,00:00:00.358000,358.0
...,...,...,...,...,...,...,...
41670,VER,10997,7,276,57.0,01:31:44.464000,94240.0
41671,VER,11059,7,277,57.0,01:31:44.596000,94372.0
41672,VER,11121,7,279,57.0,01:31:44.744000,94520.0
41673,VER,11044,7,277,57.0,01:31:44.916000,94692.0


In [21]:
df_recalculated_time["Time_lap_lbl"] = df_recalculated_time["Time_ms"].apply(lambda a: datetime.fromtimestamp(float(a)/1000.0))


In [22]:
def convert_time_lap_lbl(input):
    input = str(input)
    return "00:" + input.split(" ")[1].split(':', 1)[1]

In [23]:
df_recalculated_time["Time_lap_lbl"] = df_recalculated_time["Time_lap_lbl"].apply(lambda a: convert_time_lap_lbl(a))

In [24]:
df_recalculated_time.loc[df_recalculated_time["LapNumber"]==2.0, :]

,Driver,RPM,nGear,Speed,LapNumber,Time,Time_ms,Time_lap_lbl
743,VER,11100,7,278,2.0,00:01:37.607000,0.0,00:00:00
744,VER,11109,7,279,2.0,00:01:37.697000,90.0,00:00:00.090000
745,VER,11119,7,280,2.0,00:01:37.807000,200.0,00:00:00.200000
746,VER,11141,7,280,2.0,00:01:37.897000,290.0,00:00:00.290000
747,VER,11163,7,281,2.0,00:01:38.007000,400.0,00:00:00.400000
...,...,...,...,...,...,...,...,...
1492,VER,10992,7,276,2.0,00:03:13.408000,95801.0,00:01:35.801000
1493,VER,11015,7,276,2.0,00:03:13.517000,95910.0,00:01:35.910000
1494,VER,11039,7,276,2.0,00:03:13.568000,95961.0,00:01:35.961000
1495,VER,11068,7,277,2.0,00:03:13.677000,96070.0,00:01:36.070000


In [25]:
if CSV_SAVE:
    df_recalculated_time.to_csv(
        "../../data/assignement_01/processed/preprocessed_VER.csv",
        index=False,
    )

In [52]:
grps = df_recalculated_time.groupby("LapNumber")

results = []

for lap_no, group in grps:
    last_row = group[["LapNumber", "Time_ms"]].iloc[-1]
    results.append(last_row)

df_lap_times = pd.DataFrame(results).reset_index(drop=True)

df_lap_times["Time_ms"] = df_lap_times["Time_ms"].astype("float")

df_lap_times
# --- DIE STATISTIK-POWER ---
"""mean_time = df_lap_times['LapDuration'].mean()
std_dev = df_lap_times['LapDuration'].std()
cv = (std_dev / mean_time) * 100 
median_time = df_lap_times['LapDuration'].median()"""

"mean_time = df_lap_times['LapDuration'].mean()\nstd_dev = df_lap_times['LapDuration'].std()\ncv = (std_dev / mean_time) * 100 \nmedian_time = df_lap_times['LapDuration'].median()"

In [44]:
df_recalculated_time[["LapNumber", "Time_ms"]].iloc[-1].values

array([np.float64(57.0), '94833.0'], dtype=object)

In [ ]:
grps = df_recalculated_time.groupby("LapNumber")

results = []

for lap_no, group in grps:
    last_row = group[["LapNumber", "Time_ms"]].iloc[-1]
    results.append(last_row)

df_lap_times = pd.DataFrame(results).reset_index(drop=True)

df_lap_times["Time_ms"] = df_lap_times["Time_ms"].astype("float")

df_lap_times

df_lap_times = df_lap_times.sort_values(by="Time_ms", ascending=True)
df_lap_times

mean_time = df_lap_times['Time_ms'].mean()
std_dev = df_lap_times['Time_ms'].std()
median_time = df_lap_times['Time_ms'].median()

96419.80701754386
4164.238176076476
4.3188617617942135
95474.0


In [61]:
df_lap_times.loc[0, "Time_ms"]

np.float64(92458.0)